In [2]:
#extracting feature for one subject i.e, for subject-1 inclduing all 64 channels

import os
import numpy as np
import mne
from scipy.signal import welch

# ==========================================================
# CONFIG
# ==========================================================

SUBJECT = "S001"

EPOCH_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/processed/epochs"
FEATURE_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power"

R04_PATH = os.path.join(EPOCH_DIR, SUBJECT, f"{SUBJECT}R04_epo.fif")
R08_PATH = os.path.join(EPOCH_DIR, SUBJECT, f"{SUBJECT}R08_epo.fif")
R12_PATH = os.path.join(EPOCH_DIR, SUBJECT, f"{SUBJECT}R12_epo.fif")

SUBJECT_FEATURE_DIR = os.path.join(FEATURE_DIR, SUBJECT)
os.makedirs(SUBJECT_FEATURE_DIR, exist_ok=True)

# ==========================================================
# LOAD EPOCHS
# ==========================================================

print("Loading epochs...")

epochs_r04 = mne.read_epochs(R04_PATH, preload=True, verbose=False)
epochs_r08 = mne.read_epochs(R08_PATH, preload=True, verbose=False)
epochs_r12 = mne.read_epochs(R12_PATH, preload=True, verbose=False)

print("R04:", len(epochs_r04))
print("R08:", len(epochs_r08))
print("R12:", len(epochs_r12))

# ==========================================================
# CHECK EVENT IDS
# ==========================================================

print("\nEvent IDs:")
print(epochs_r04.event_id)

# ==========================================================
# TRAIN = R04 + R08
# ==========================================================

train_epochs = mne.concatenate_epochs([epochs_r04, epochs_r08])

print("\nTraining epochs:", len(train_epochs))
print("Testing epochs :", len(epochs_r12))

# ==========================================================
# FEATURE EXTRACTION
# ==========================================================

def extract_bandpower_features(epochs):
    """
    Extract:
        Mu band   : 8-12 Hz
        Beta band : 13-30 Hz

    Output:
        (n_epochs, n_channels * 2)
    """

    data = epochs.get_data()
    sfreq = epochs.info["sfreq"]

    all_features = []

    for epoch in data:

        epoch_features = []

        for channel in epoch:

            nperseg = min(256, len(channel))

            freqs, psd = welch(
                channel,
                fs=sfreq,
                nperseg=nperseg
            )

            mu_mask = (freqs >= 8) & (freqs <= 12)
            beta_mask = (freqs >= 13) & (freqs <= 30)

            mu_power = np.trapezoid(
                psd[mu_mask],
                freqs[mu_mask]
            )

            beta_power = np.trapezoid(
                psd[beta_mask],
                freqs[beta_mask]
            )

            epoch_features.extend([
                mu_power,
                beta_power
            ])

        all_features.append(epoch_features)

    return np.array(all_features)

# ==========================================================
# EXTRACT FEATURES
# ==========================================================

print("\nExtracting training features...")
X_train = extract_bandpower_features(train_epochs)

print("Extracting test features...")
X_test = extract_bandpower_features(epochs_r12)

# ==========================================================
# LABELS
# ==========================================================

y_train = train_epochs.events[:, -1]
y_test = epochs_r12.events[:, -1]

# ==========================================================
# SAVE FEATURES
# ==========================================================

np.save(
    os.path.join(
        SUBJECT_FEATURE_DIR,
        "S001_train_bandpower_X.npy"
    ),
    X_train
)

np.save(
    os.path.join(
        SUBJECT_FEATURE_DIR,
        "S001_train_bandpower_y.npy"
    ),
    y_train
)

np.save(
    os.path.join(
        SUBJECT_FEATURE_DIR,
        "S001_test_bandpower_X.npy"
    ),
    X_test
)

np.save(
    os.path.join(
        SUBJECT_FEATURE_DIR,
        "S001_test_bandpower_y.npy"
    ),
    y_test
)

# save label mapping
np.save(
    os.path.join(
        SUBJECT_FEATURE_DIR,
        "S001_event_id.npy"
    ),
    epochs_r04.event_id,
    allow_pickle=True
)

# ==========================================================
# SUMMARY
# ==========================================================

print("\n========== SUMMARY ==========")

print("X_train shape :", X_train.shape)
print("y_train shape :", y_train.shape)

print("X_test shape  :", X_test.shape)
print("y_test shape  :", y_test.shape)

print("\nTrain labels:")
print(dict(zip(*np.unique(y_train, return_counts=True))))

print("\nTest labels:")
print(dict(zip(*np.unique(y_test, return_counts=True))))

print("\nSaved to:")
print(SUBJECT_FEATURE_DIR)
print(epochs_r04.event_id)

Loading epochs...
R04: 30
R08: 30
R12: 30

Event IDs:
{'rest': 1, 'left': 2, 'right': 3}
Not setting metadata
60 matching events found
No baseline correction applied

Training epochs: 60
Testing epochs : 30

Extracting training features...


/var/folders/2p/pvb41hns2szdm00nww51m7r40000gn/T/ipykernel_77202/929209437.py:49: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  train_epochs = mne.concatenate_epochs([epochs_r04, epochs_r08])


Extracting test features...

========== SUMMARY ==========
X_train shape : (60, 128)
y_train shape : (60,)
X_test shape  : (30, 128)
y_test shape  : (30,)

Train labels:
{np.int32(1): np.int64(30), np.int32(2): np.int64(16), np.int32(3): np.int64(14)}

Test labels:
{np.int32(1): np.int64(15), np.int32(2): np.int64(7), np.int32(3): np.int64(8)}

Saved to:
/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power/S001
{'rest': 1, 'left': 2, 'right': 3}


In [ ]:
#feature extarction from the C3,Cz and C4 for one subject

import os
import numpy as np
import mne
from scipy.signal import welch

# ==========================================================
# Paths
# ==========================================================
EPOCH_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/processed/epochs"
FEATURE_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power_c3_cz_c4"

R04_PATH = f"{EPOCH_DIR}/S001/S001R04_epo.fif"
R08_PATH = f"{EPOCH_DIR}/S001/S001R08_epo.fif"
R12_PATH = f"{EPOCH_DIR}/S001/S001R12_epo.fif"

SUBJECT_FEATURE_DIR = os.path.join(FEATURE_DIR, "S001")
os.makedirs(SUBJECT_FEATURE_DIR, exist_ok=True)

# Motor cortex channels only — C3 (left motor cortex, contralateral to
# right-hand movement/imagery), C4 (right motor cortex, contralateral to
# left-hand movement/imagery), Cz (vertex, central reference point,
# relevant for foot/bilateral imagery). Restricting to these 3 channels
# instead of all 64 is grounded in the literature on contralateral mu
# rhythm lateralization (Pfurtscheller et al. 2006; see docs/decisions.md).
MOTOR_CHANNELS = ["C3..", "Cz..", "C4.."]

# Welch PSD window length — fixed explicitly so frequency resolution
# stays consistent across all runs/subjects regardless of epoch length.
NPERSEG = 256

# ==========================================================
# Load epochs
# ==========================================================
epochs_r04 = mne.read_epochs(R04_PATH, preload=True, verbose=False)
epochs_r08 = mne.read_epochs(R08_PATH, preload=True, verbose=False)
epochs_r12 = mne.read_epochs(R12_PATH, preload=True, verbose=False)

# Sanity check: all three runs should share the same event_id mapping
assert epochs_r04.event_id == epochs_r08.event_id == epochs_r12.event_id, (
    "event_id mapping differs between runs — check epoch extraction step"
)
EVENT_ID_MAP = epochs_r04.event_id
print("Event ID mapping (name -> code):", EVENT_ID_MAP)

# Sanity check: confirm motor channels actually exist in this dataset's
# channel list before going further. PhysioNet channel names sometimes
# carry trailing periods (e.g. "C3.") depending on the export — handle
# both forms rather than assuming exact match.
def resolve_channel_names(available_chs, wanted):
    resolved = []
    for ch in wanted:
        if ch in available_chs:
            resolved.append(ch)
        elif f"{ch}." in available_chs:
            resolved.append(f"{ch}.")
        else:
            raise ValueError(
                f"Channel '{ch}' not found in dataset channels: {available_chs}"
            )
    return resolved

available_channels = epochs_r04.info["ch_names"]
RESOLVED_MOTOR_CHANNELS = resolve_channel_names(available_channels, MOTOR_CHANNELS)
print("Resolved motor channel names:", RESOLVED_MOTOR_CHANNELS)

# ==========================================================
# Restrict all epoch objects to motor channels only
# ==========================================================
epochs_r04 = epochs_r04.copy().pick(RESOLVED_MOTOR_CHANNELS)
epochs_r08 = epochs_r08.copy().pick(RESOLVED_MOTOR_CHANNELS)
epochs_r12 = epochs_r12.copy().pick(RESOLVED_MOTOR_CHANNELS)

# ==========================================================
# Concatenate training epochs
# ==========================================================
train_epochs = mne.concatenate_epochs([epochs_r04, epochs_r08])

# ==========================================================
# Feature Extraction Function
# ==========================================================
def extract_bandpower_features(epochs, nperseg=NPERSEG):
    """
    Extract mu (8-12 Hz) and beta (13-30 Hz) band power for each of the
    motor cortex channels (C3, Cz, C4) of every epoch, using Welch's
    method for PSD estimation.

    Returns array of shape (n_epochs, n_channels * 2) = (n_epochs, 6),
    where each channel contributes [mu_power, beta_power], in channel
    order C3, Cz, C4.
    """
    X = epochs.get_data()  # (n_epochs, n_channels=3, n_samples)
    sfreq = epochs.info["sfreq"]

    if X.shape[-1] < nperseg:
        raise ValueError(
            f"Epoch length ({X.shape[-1]} samples) is shorter than "
            f"nperseg ({nperseg}). Fix nperseg or check epoching tmin/tmax."
        )

    features = []
    for epoch in X:
        epoch_features = []
        for channel_signal in epoch:
            freqs, psd = welch(channel_signal, fs=sfreq, nperseg=nperseg)

            mu_mask = (freqs >= 8) & (freqs <= 12)
            beta_mask = (freqs >= 13) & (freqs <= 30)

            mu_power = np.trapezoid(psd[mu_mask], freqs[mu_mask])
            beta_power = np.trapezoid(psd[beta_mask], freqs[beta_mask])

            epoch_features.extend([mu_power, beta_power])
        features.append(epoch_features)
    return np.array(features)

# ==========================================================
# Extract Features
# ==========================================================
X_train = extract_bandpower_features(train_epochs)
X_test = extract_bandpower_features(epochs_r12)

# ==========================================================
# Labels
# ==========================================================
y_train = train_epochs.events[:, -1]
y_test = epochs_r12.events[:, -1]

# ==========================================================
# Save Features
# ==========================================================
np.save(os.path.join(SUBJECT_FEATURE_DIR, "S001_train_bandpower_X.npy"), X_train)
np.save(os.path.join(SUBJECT_FEATURE_DIR, "S001_train_bandpower_y.npy"), y_train)
np.save(os.path.join(SUBJECT_FEATURE_DIR, "S001_test_bandpower_X.npy"), X_test)
np.save(os.path.join(SUBJECT_FEATURE_DIR, "S001_test_bandpower_y.npy"), y_test)
np.save(
    os.path.join(SUBJECT_FEATURE_DIR, "S001_event_id_map.npy"),
    EVENT_ID_MAP,
    allow_pickle=True,
)
np.save(
    os.path.join(SUBJECT_FEATURE_DIR, "S001_channel_order.npy"),
    np.array(RESOLVED_MOTOR_CHANNELS),
)

# ==========================================================
# Check
# ==========================================================
print("\nFeature vector layout: [C3_mu, C3_beta, Cz_mu, Cz_beta, C4_mu, C4_beta]")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("\nLabel distribution (train):", dict(zip(*np.unique(y_train, return_counts=True))))
print("Label distribution (test):", dict(zip(*np.unique(y_test, return_counts=True))))
print("\nSaved feature files to:", SUBJECT_FEATURE_DIR)

In [ ]:
#feature for 15 channels

import os
import numpy as np
import mne
from scipy.signal import welch

# ==========================================================
# Paths
# ==========================================================
EPOCH_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/processed/epochs"
FEATURE_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power_15"

R04_PATH = f"{EPOCH_DIR}/S001/S001R04_epo.fif"
R08_PATH = f"{EPOCH_DIR}/S001/S001R08_epo.fif"
R12_PATH = f"{EPOCH_DIR}/S001/S001R12_epo.fif"

SUBJECT_FEATURE_DIR = os.path.join(FEATURE_DIR, "S001")
os.makedirs(SUBJECT_FEATURE_DIR, exist_ok=True)

# Motor cortex region: C3/Cz/C4 plus immediate neighbors (FC and CP rows).
# This is a moderate middle ground between the full 64-channel set and
# the narrow 3-channel C3/Cz/C4-only set — see docs/decisions.md for the
# comparison rationale (full-64 vs C3/Cz/C4 vs this 15-channel set).
#
# NOTE: written WITHOUT trailing dots here — exact dataset naming
# (e.g. "C3.." vs "C3") is resolved automatically below, case-insensitively,
# so this list only needs to express the canonical 10-20 channel names once.
MOTOR_CHANNELS = [
    "FC3", "FC1", "FCz", "FC2", "FC4",
    "C3", "C1", "Cz", "C2", "C4",
    "CP3", "CP1", "CPz", "CP2", "CP4",
]

# Hard guard against accidental duplicates in the list above —
# this is exactly the bug that broke the previous version (C3/Cz/C4
# each appeared twice), so fail loudly and early if it happens again.
assert len(MOTOR_CHANNELS) == len(set(MOTOR_CHANNELS)), (
    f"Duplicate entries found in MOTOR_CHANNELS: {MOTOR_CHANNELS}"
)

# Welch PSD window length — fixed explicitly so frequency resolution
# stays consistent across all runs/subjects regardless of epoch length.
NPERSEG = 256

# ==========================================================
# Load epochs
# ==========================================================
epochs_r04 = mne.read_epochs(R04_PATH, preload=True, verbose=False)
epochs_r08 = mne.read_epochs(R08_PATH, preload=True, verbose=False)
epochs_r12 = mne.read_epochs(R12_PATH, preload=True, verbose=False)

# Sanity check: all three runs should share the same event_id mapping
assert epochs_r04.event_id == epochs_r08.event_id == epochs_r12.event_id, (
    "event_id mapping differs between runs — check epoch extraction step"
)
EVENT_ID_MAP = epochs_r04.event_id
print("Event ID mapping (name -> code):", EVENT_ID_MAP)

# ==========================================================
# Channel resolution — handles trailing-dot naming and case
# differences (PhysioNet EDF channel names are inconsistent about
# both), and fails loudly + early if a channel truly isn't found,
# rather than silently dropping it.
# ==========================================================
def resolve_channel_names(available_chs, wanted):
    """
    Match each wanted channel name against the actual channel names
    in the dataset, trying exact match, then case-insensitive match,
    then case-insensitive match with trailing dot(s) appended.
    Raises if a channel can't be resolved, and raises if the same
    actual channel gets matched twice (would cause pick() to fail).
    """
    lookup = {ch.lower().rstrip("."): ch for ch in available_chs}
    resolved = []
    seen_actual = set()

    for ch in wanted:
        key = ch.lower().rstrip(".")
        if key not in lookup:
            raise ValueError(
                f"Channel '{ch}' not found in dataset channels: {available_chs}"
            )
        actual = lookup[key]
        if actual in seen_actual:
            raise ValueError(
                f"Channel '{ch}' resolved to '{actual}', which was already "
                f"matched by an earlier entry — check for duplicates in the "
                f"requested channel list."
            )
        seen_actual.add(actual)
        resolved.append(actual)

    return resolved

available_channels = epochs_r04.info["ch_names"]
RESOLVED_MOTOR_CHANNELS = resolve_channel_names(available_channels, MOTOR_CHANNELS)
print(f"Resolved {len(RESOLVED_MOTOR_CHANNELS)} motor channel names:")
print(RESOLVED_MOTOR_CHANNELS)

# ==========================================================
# Restrict all epoch objects to motor channels only
# ==========================================================
epochs_r04 = epochs_r04.copy().pick(RESOLVED_MOTOR_CHANNELS)
epochs_r08 = epochs_r08.copy().pick(RESOLVED_MOTOR_CHANNELS)
epochs_r12 = epochs_r12.copy().pick(RESOLVED_MOTOR_CHANNELS)

# ==========================================================
# Concatenate training epochs
# ==========================================================
train_epochs = mne.concatenate_epochs([epochs_r04, epochs_r08])

# ==========================================================
# Feature Extraction Function
# ==========================================================
def extract_bandpower_features(epochs, nperseg=NPERSEG):
    """
    Extract mu (8-12 Hz) and beta (13-30 Hz) band power for each
    channel of every epoch, using Welch's method for PSD estimation.

    Returns array of shape (n_epochs, n_channels * 2), where each
    channel contributes [mu_power, beta_power], in the channel order
    given by RESOLVED_MOTOR_CHANNELS.
    """
    X = epochs.get_data()  # (n_epochs, n_channels, n_samples)
    sfreq = epochs.info["sfreq"]

    if X.shape[-1] < nperseg:
        raise ValueError(
            f"Epoch length ({X.shape[-1]} samples) is shorter than "
            f"nperseg ({nperseg}). Fix nperseg or check epoching tmin/tmax."
        )

    features = []
    for epoch in X:
        epoch_features = []
        for channel_signal in epoch:
            freqs, psd = welch(channel_signal, fs=sfreq, nperseg=nperseg)

            mu_mask = (freqs >= 8) & (freqs <= 12)
            beta_mask = (freqs >= 13) & (freqs <= 30)

            mu_power = np.trapezoid(psd[mu_mask], freqs[mu_mask])
            beta_power = np.trapezoid(psd[beta_mask], freqs[beta_mask])

            epoch_features.extend([mu_power, beta_power])
        features.append(epoch_features)
    return np.array(features)

# ==========================================================
# Extract Features
# ==========================================================
X_train = extract_bandpower_features(train_epochs)
X_test = extract_bandpower_features(epochs_r12)

# ==========================================================
# Labels
# ==========================================================
y_train = train_epochs.events[:, -1]
y_test = epochs_r12.events[:, -1]

# ==========================================================
# Save Features
# ==========================================================
np.save(os.path.join(SUBJECT_FEATURE_DIR, "S001_train_bandpower_X.npy"), X_train)
np.save(os.path.join(SUBJECT_FEATURE_DIR, "S001_train_bandpower_y.npy"), y_train)
np.save(os.path.join(SUBJECT_FEATURE_DIR, "S001_test_bandpower_X.npy"), X_test)
np.save(os.path.join(SUBJECT_FEATURE_DIR, "S001_test_bandpower_y.npy"), y_test)
np.save(
    os.path.join(SUBJECT_FEATURE_DIR, "S001_event_id_map.npy"),
    EVENT_ID_MAP,
    allow_pickle=True,
)
np.save(
    os.path.join(SUBJECT_FEATURE_DIR, "S001_channel_order.npy"),
    np.array(RESOLVED_MOTOR_CHANNELS),
)

# ==========================================================
# Check
# ==========================================================
feature_layout = []
for ch in RESOLVED_MOTOR_CHANNELS:
    feature_layout.extend([f"{ch}_mu", f"{ch}_beta"])
print("\nFeature vector layout:", feature_layout)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("\nLabel distribution (train):", dict(zip(*np.unique(y_train, return_counts=True))))
print("Label distribution (test):", dict(zip(*np.unique(y_test, return_counts=True))))
print("\nSaved feature files to:", SUBJECT_FEATURE_DIR)